In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Itakda ang optimization level ng transpiler

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na kinakailangan.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Ang mga tunay na quantum device ay naaapektuhan ng ingay at mga gate error, kaya ang pag-optimize ng mga circuit para mabawasan ang kanilang depth at bilang ng gate ay malaki ang maiaambag sa pagpapabuti ng mga resultang makukuha mula sa pagpapatakbo ng mga circuit na iyon.
Ang function na [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) ay may isang kinakailangang positional argument, ang `optimization_level`, na kumokontrol sa gaano kahirap subukan ng Transpiler na i-optimize ang mga circuit. Ang argument na ito ay maaaring maging integer na may halagang 0, 1, 2, o 3.
Ang mas mataas na optimization level ay nagbibigay ng mas na-optimize na mga circuit, ngunit mas matagal ang oras ng pag-compile.
Ang sumusunod na talahanayan ay nagpapaliwanag ng mga optimization na isinasagawa sa bawat setting.

<table>
  <thead>
    <tr>
      <th>Optimization Level</th>
      <th>Paglalarawan</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Walang optimization: karaniwang ginagamit para sa hardware characterization
        - Pangunahing pagsasalin
        - Layout/Routing: `TrivialLayout`, kung saan pinipili nito ang parehong pisikal na bilang ng qubit gaya ng virtual at naglalagay ng mga SWAP para gumana ito (gamit ang `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Magaang na optimization:
        -   Layout/Routing: Unang sinusubukan ang Layout gamit ang `TrivialLayout`. Kung kailangan ng karagdagang SWAP, hinahanap ang layout na may pinakamaliit na bilang ng SWAP sa pamamagitan ng `SabreSwap`, pagkatapos ay gumagamit ito ng `VF2LayoutPostLayout` upang subukang piliin ang pinakamainam na mga qubit sa graph.
        -   `InverseCancellation`
        -   1Q gate optimization
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Katamtamang optimization:
          - Layout/Routing: Optimization level 1 (walang trivial) + heuristic na na-optimize na may mas malalim na paghahanap at mas maraming pagsubok ng optimization function. Dahil hindi ginagamit ang `TrivialLayout`, walang pagtatangkang gamitin ang parehong pisikal at virtual na bilang ng qubit.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Mataas na Optimization:
        - Optimization level 2 + heuristic na mas na-optimize pa sa layout/routing na may mas malaking pagsisikap/pagsubok
        - Muling synthesis ng mga two-qubit block gamit ang [Cartan's KAK Decomposition](https://arxiv.org/abs/quant-ph/0507171).
        - Mga unitarity-breaking pass:
          * `OptimizeSwapBeforeMeasure`: Inililipat ang mga measurement para maiwasan ang mga SWAP
          * `RemoveDiagonalGatesBeforeMeasure`: Inaalis ang mga gate bago ang mga measurement na hindi makakaapekto sa mga measurement
      </td>
    </tr>
  </tbody>
</table>

## Ang optimization level sa gawa
Dahil ang mga two-qubit gate ay karaniwang pinaka-malaking pinagmumulan ng mga error, maaari nating tantiyahin ang "hardware efficiency" ng transpilation sa pamamagitan ng pagbibilang ng bilang ng mga two-qubit gate sa resultang circuit.
Dito, susubukan natin ang iba't ibang optimization level sa isang input circuit na binubuo ng isang random unitary na sinusundan ng isang SWAP gate.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Gagamitin natin ang `FakeSherbrooke` mock backend sa ating mga halimbawa. Una, i-transpile natin gamit ang optimization level 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Ang na-transpile na circuit ay may anim na two-qubit ECR gate.

Ulitin para sa optimization level 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Ang na-transpile na circuit ay mayroon pa ring anim na ECR gate, ngunit nabawasan ang bilang ng mga single-qubit gate.

Ulitin para sa optimization level 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

Ito ay nagbibigay ng parehong resulta gaya ng optimization level 1. Tandaan na ang pagdaragdag ng antas ng optimization ay hindi palaging nagbabago ng resulta.

Ulitin muli, gamit ang optimization level 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Ngayon, tatlo na lang ang ECR gate. Nakukuha natin ang resultang ito dahil sa optimization level 3, sinusubukan ng Qiskit na muling i-synthesize ang mga two-qubit block ng mga gate, at anumang two-qubit gate ay maaaring ipatupad gamit ang hindi hihigit sa tatlong ECR gate. Maaari pa nating mabawasan ang bilang ng ECR gate kung itatakda natin ang `approximation_degree` sa halagang mas mababa sa 1, na nagpapahintulot sa Transpiler na gumawa ng mga approximation na maaaring magdulot ng kaunting error sa gate decomposition (tingnan ang [Mga karaniwang ginagamit na parameter para sa transpilation](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Ang circuit na ito ay may dalawang ECR gate lamang, ngunit ito ay isang approximate circuit. Para maunawaan kung paano naiiba ang epekto nito mula sa eksaktong circuit, maaari nating kalkulahin ang fidelity sa pagitan ng unitary operator na ipinapatupad ng circuit na ito, at ng eksaktong unitary. Bago gawin ang kalkulasyon, unang binabawasan natin ang na-transpile na circuit, na naglalaman ng 127 qubit, pababa sa isang circuit na naglalaman lamang ng mga aktibong qubit, na dalawa ang bilang.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>